# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohinaRustamova/lyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Distributions before any decisions. Looking at gsc_impressions, gsc_clicks, and CTR before testing any signal, the same discipline as checking eligibility before comparing CTR in ML-07. All three fields show heavy right tails, a small number of pages carry most of the volume, most pages get very little traffic.

In [14]:
# ---- Setup: load data and rebuild feature vector (matches ML-04 pattern) ----
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL, REPO_DIR = "https://github.com/MohinaRustamova/flyrank-ml-internship", "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

%pip -q install duckdb

import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

if IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    from getpass import getpass
    HF_TOKEN = getpass("HF_TOKEN: ")

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
DIMC = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

DECISION_DATE = "2026-03-31"
decision_ts = pd.Timestamp(DECISION_DATE)

# --- Current + prior 30-day windows ---
df = con.sql(f"""
    WITH current_win AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS gsc_impressions,
               SUM(gsc_clicks) AS gsc_clicks,
               AVG(gsc_avg_position) AS gsc_avg_position
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-02' AND DATE '2026-03-31'
          AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    ),
    prior_win AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_prior30
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-01-31' AND DATE '2026-03-01'
          AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT c.content_hash_id,
           c.gsc_impressions, c.gsc_clicks, c.gsc_avg_position,
           p.impressions_prior30,
           CASE WHEN c.gsc_impressions < p.impressions_prior30 * 0.8
                THEN 1 ELSE 0 END AS is_declining_label
    FROM current_win c
    JOIN prior_win p USING (content_hash_id)
    WHERE p.impressions_prior30 > 0
""").df()

# --- content_age_days ---
dim_content2 = con.sql(f"SELECT content_hash_id, content_created_date FROM {DIMC}").df()
df = df.merge(dim_content2, on="content_hash_id", how="left")
df["content_created_date"] = pd.to_datetime(df["content_created_date"])
df["content_age_days"] = (decision_ts - df["content_created_date"]).dt.days

print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(135113, 8)


,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,impressions_prior30,is_declining_label,content_created_date,content_age_days
0,content_e67934818ca184a1,1072.0,2.0,15.667524,320.0,0,2025-04-14,351
1,content_9634c35544bcc47b,313.0,1.0,11.978349,411.0,1,2025-04-14,351
2,content_4ece07fdea783709,653.0,0.0,17.465739,629.0,0,2025-04-14,351
3,content_4d9b25ca95147676,1028.0,18.0,4.869805,1421.0,1,2025-04-14,351
4,content_2dae25d4660d074a,1366.0,1.0,24.638455,1209.0,0,2025-04-14,351


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"]

print(df[["gsc_impressions", "gsc_clicks", "ctr"]].describe())
print()
print("Zero-click rows:", (df["gsc_clicks"] == 0).sum(), "of", len(df),
      f"({(df['gsc_clicks'] == 0).mean():.1%})")
print("Top 1% of impressions accounts for this share of total impressions:",
      f"{df.nlargest(int(len(df)*0.01), 'gsc_impressions')['gsc_impressions'].sum() / df['gsc_impressions'].sum():.1%}")

       gsc_impressions     gsc_clicks            ctr
count    135113.000000  135113.000000  135113.000000
mean       1880.818263       5.328747       0.004034
std        5958.792080      29.361882       0.030578
min           1.000000       0.000000       0.000000
25%          40.000000       0.000000       0.000000
50%         258.000000       0.000000       0.000000
75%        1373.000000       2.000000       0.002309
max      613219.000000    5631.000000       1.000000

Zero-click rows: 77610 of 135113 (57.4%)
Top 1% of impressions accounts for this share of total impressions: 23.1%


Distributions: heavy right tail confirmed. Median impressions (258) and median clicks (0) sit far below the mean (1,881 impressions, 5.3 clicks), and 57.4% of rows have zero clicks entirely. The top 1% of pages by impressions account for 23.1% of all impressions in the dataset. This isn't noise, it's the real shape of a large content inventory: most pages barely get seen, a small number carry most of the traffic. This is why an impressions floor was necessary before any CTR comparison in Section 2 (a median computed across mostly-zero rows isn't measuring anything real).

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Test #1: Content age vs decline rate

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
age_bins = [0, 90, 180, 365, float("inf")]
age_labels = ["0-3mo", "3-6mo", "6-12mo", "12+mo"]
df["age_bucket"] = pd.cut(df["content_age_days"], bins=age_bins, labels=age_labels)

test1 = (
    df.groupby("age_bucket", observed=True)
      .agg(n=("content_hash_id", "count"), share_declining=("is_declining_label", "mean"))
      .assign(share_declining=lambda x: (x["share_declining"] * 100).round(1))
)
print(test1)

                n  share_declining
age_bucket                        
0-3mo       30660             20.2
3-6mo       23608             26.6
6-12mo      60894             25.7
12+mo       19951             21.8


Verdict: MIXED. Decline rate doesn't climb steadily with age, it rises then falls in the oldest bucket. Age matters but isn't a clean standalone predictor (same finding as ML-07 Section 1).

Test #2: CTR vs position peers

In [17]:
MIN_IMPRESSIONS = 100
eligible = df[df["gsc_impressions"] >= MIN_IMPRESSIONS].copy()

pos_bins = [0, 3, 6, 10, 20, float("inf")]
pos_labels = ["1-3", "3-6", "6-10", "10-20", "20+"]
eligible["position_bucket"] = pd.cut(eligible["gsc_avg_position"], bins=pos_bins, labels=pos_labels)

peer_median = eligible.groupby("position_bucket", observed=True)["ctr"].transform("median")
eligible["ctr_below_peers"] = eligible["ctr"] < (peer_median * 0.5)

test2 = (
    eligible.groupby("position_bucket", observed=True)
      .agg(n=("content_hash_id", "count"), median_ctr=("ctr", "median"),
           share_far_below=("ctr_below_peers", "mean"))
      .assign(median_ctr=lambda x: (x["median_ctr"] * 100).round(2),
              share_far_below=lambda x: (x["share_far_below"] * 100).round(1))
)
print(test2)

                     n  median_ctr  share_far_below
position_bucket                                    
1-3               6813        0.21             32.5
3-6              20182        0.21             31.9
6-10             19425        0.15             39.0
10-20            18887        0.09             45.2
20+              21094        0.00              0.0


Verdict: CONFIRMED, with a documented limit at position 20+, where peer median CTR is 0 and the comparison becomes mathematically undefined (same result as ML-07 Section 1, an eligibility floor of 100 impressions was required first).

Test #3: Volume vs flag-worthiness

In [18]:
# Does high-impression content get flagged as underperforming more often, just because it's more visible?
vol_bins = [0, 100, 1000, 10000, float("inf")]
vol_labels = ["<100", "100-1K", "1K-10K", "10K+"]
eligible["volume_bucket"] = pd.cut(eligible["gsc_impressions"], bins=vol_bins, labels=vol_labels)

test3 = (
    eligible.groupby("volume_bucket", observed=True)
      .agg(n=("content_hash_id", "count"), share_flagged=("ctr_below_peers", "mean"))
      .assign(share_flagged=lambda x: (x["share_flagged"] * 100).round(1))
)
print(test3)

                   n  share_flagged
volume_bucket                      
<100             185           48.1
100-1K         46315           38.4
1K-10K         34458           18.1
10K+            5444           12.1


Verdict: OPPOSITE. The question was whether high-impression content gets flagged more often just because it's more visible. The data shows the reverse, flag rate falls as volume rises. High-traffic pages (10K+ impressions) are flagged only 12.1% of the time, versus 38-48% for lower-volume pages. This makes sense on reflection: high-traffic pages have earned their traffic by performing reasonably well, and the peer-median comparison itself uses mostly high-volume pages to set the median, so the biggest pages are being compared against others like them, not inflated by their own visibility. This is a useful negative result, it rules out a real risk (that the score could just be a proxy for "big page, therefore flagged") rather than genuinely measuring underperformance.

Note: <100 bucket only has 185 rows because most of the eligible set already excludes anything below the 100-impression floor, this row is here for completeness but shouldn't be read as a reliable rate given the tiny sample.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*


Flag-linked test: CTR vs position peers (Test #2). This directly underlies FlyRank's real CTR_BELOW_POSITION_PEERS flag from the session. The data supports the rule's core assumption, median CTR clearly falls as position worsens (0.21% at position 1-3 down to 0.09% at position 10-20), confirming that position and CTR are genuinely linked. The one caveat is position 20+, where peer CTR itself is 0, making "below peers" mathematically undefined there, so the flag should never fire on position 20+ content, that's a data limit, not a content problem.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*


A content team using these flags should trust the CTR-vs-position signal (Test #2, CONFIRMED) as the primary trigger for a snippet review, but treat "old content" alone (Test #1, MIXED) as a weak, secondary signal, not a standalone reason to act. Test #3 is reassuring: being flagged isn't just a side effect of having more traffic, low-traffic pages are actually flagged more often, so the CTR flag is measuring real underperformance relative to genuine peers, not just penalizing visibility. Anything below the 100-impression floor, or at position 20+, should be excluded from CTR-based flags entirely, the comparison isn't reliable there.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.